# Runtime scaling of SciPy L-BFGS-B

This notebook investigates a possible runtime-scaling issue in `scipy.optimize.minimize(method="L-BFGS-B")`. The comparison methods are the line-search L-BFGS implementation (`qnlab.solver.qn_line`) and the regularized/non-monotone implementation (`qnlab.solver.qn_ntrqn`) in this repository.

The question is deliberately about **wall-clock overhead**, not which method reaches the smallest objective value. We test two distinct scaling questions:

1. elapsed time versus the requested iteration budget, up to 10,000 iterations;
2. elapsed time versus problem dimension at a fixed iteration budget.

A result is potentially issue-worthy only if it is reproducible across fresh processes/machines and remains visible after accounting for objective/gradient evaluation cost. The notebook therefore records versions, evaluation counts, exit messages, and empirical log--log slopes; it does not assume in advance that SciPy is slower or superlinear.

SciPy option definitions: [`minimize(method='L-BFGS-B')`](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-lbfgsb.html).


## Reproduction protocol

Run this notebook from the repository root in a fresh kernel. Close CPU-heavy applications and, for an issue report, repeat the full experiment in at least two independent processes. All solvers receive an analytic gradient, use L-BFGS memory `m=10`, run without callbacks, and have convergence tolerances disabled as far as their public interfaces allow. The zero-chain construction below is unconstrained, so SciPy's L-BFGS-B is being used with `bounds=None`.

Each timed sample is a fresh optimization. This costs more than collecting timestamps in a callback, but avoids charging different callback implementations to different solvers. The execution order is rotated between repeats to reduce systematic thermal/frequency bias. Medians are used for plots.

> The complete default run can take several minutes. Reduce `REPEATS` while developing, but use at least 3 repeats for a report.


In [ ]:
from __future__ import annotations

import os
import platform
import sys
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.optimize import minimize

# Make the cell work when Jupyter starts in either the repository root or notebooks/.
REPO_ROOT = Path.cwd() if (Path.cwd() / "qnlab").is_dir() else Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qnlab.parameter import LineParameter, NTRQNParameter
from qnlab.problem.base import BaseProblem
from qnlab.solver.qn_line import qn_line
from qnlab.solver.qn_ntrqn import qn_ntrqn
from qnlab.util.method import Method

ENVIRONMENT = {
    "python": sys.version.replace("\n", " "),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "platform": platform.platform(),
    "processor": platform.processor() or "not reported",
    "cpu_count": os.cpu_count(),
}
pd.Series(ENVIRONMENT, name="value").to_frame()

## A smooth zero-chain problem

For dimension $n$, define

$$
f(x)=\frac12 x_1^2-x_1+\frac12\sum_{i=1}^{n-1}(x_i-x_{i+1})^2.
$$

This is a strictly convex, ill-conditioned quadratic with minimizer $x^*=(1,\ldots,1)$. At $x_0=0$, only the first gradient component is nonzero. For methods whose new directions lie in the span generated by past gradients and L-BFGS pairs, information can advance through the chain only gradually. Taking `dimension > max_iterations` therefore prevents the algorithm from simply exhausting the chain before the 10,000-iteration measurement. Both the function and gradient below cost $O(n)$ and allocate only a few vectors.


In [ ]:
class ZeroChainQuadratic(BaseProblem):
    """Nesterov-style smooth zero-chain quadratic with x*=ones(n)."""

    def __init__(self, n: int):
        super().__init__(name="ZeroChainQuadratic", n=n, x0=np.zeros(n))
        self.x_opt = np.ones(n)

    def _f(self, x: np.ndarray) -> np.float64:
        differences = x[:-1] - x[1:]
        return np.float64(0.5 * x[0] ** 2 - x[0] + 0.5 * differences @ differences)

    def _g(self, x: np.ndarray) -> np.ndarray:
        differences = x[:-1] - x[1:]
        gradient = np.zeros_like(x)
        gradient[0] = x[0] - 1.0
        gradient[:-1] += differences
        gradient[1:] -= differences
        return gradient


# Sanity checks for the formula and zero-chain structure.
_problem = ZeroChainQuadratic(8)
assert np.isclose(_problem.f(_problem.x_opt, count=False), -0.5)
assert np.allclose(_problem.g(_problem.x_opt, count=False), 0.0)
assert np.array_equal(np.flatnonzero(_problem.g(_problem.x0, count=False)), [0])

## Benchmark implementation

`ftol=0` and `gtol=0` are used to make the iteration cap, rather than ordinary convergence tolerances, control termination. Evaluation limits are set comfortably above the iteration limit. The recorded `nit` is available directly from SciPy; for qnlab, an iteration-limit return code together with the requested budget verifies the intended exit. Results with another exit reason must be investigated rather than silently plotted as equal-work samples.


In [ ]:
MEMORY = 10
ITERATION_DIMENSION = 12_000  # Must exceed max(ITERATION_BUDGETS).
ITERATION_BUDGETS = np.array([100, 200, 500, 1_000, 2_000, 5_000, 10_000])
REPEATS = 3

DIMENSION_ITERATIONS = 500
DIMENSIONS = np.array([1_000, 2_000, 5_000, 10_000, 20_000, 50_000])
DIMENSION_REPEATS = 3

assert ITERATION_DIMENSION > ITERATION_BUDGETS.max()
assert DIMENSIONS.min() > DIMENSION_ITERATIONS
SOLVERS = ("SciPy L-BFGS-B", "qnlab qn_line", "qnlab qn_ntrqn")

In [ ]:
@dataclass
class TimingResult:
    solver: str
    dimension: int
    requested_iterations: int
    repeat: int
    elapsed_seconds: float
    iterations: int | None
    function_calls: int
    gradient_calls: int
    final_objective: float
    exit_status: str


def run_once(
    solver: str,
    dimension: int,
    max_iterations: int,
    repeat: int,
    memory: int = MEMORY,
    problem_factory=ZeroChainQuadratic,
) -> TimingResult:
    problem = problem_factory(dimension)
    evaluation_limit = 50 * max_iterations + 100

    start = time.perf_counter()
    if solver == "SciPy L-BFGS-B":
        result = minimize(
            problem.f,
            problem.x0.copy(),
            jac=problem.g,
            method="L-BFGS-B",
            bounds=None,
            callback=None,
            options={
                "maxcor": memory,
                "maxiter": max_iterations,
                "maxfun": evaluation_limit,
                "ftol": 0.0,
                "gtol": 0.0,
                "maxls": 40,
            },
        )
        elapsed = time.perf_counter() - start
        iterations = int(result.nit)
        final_objective = float(result.fun)
        exit_status = str(result.message)
    elif solver == "qnlab qn_line":
        method = Method(base="Line", store="raw", secant="raw", update="bfgs")
        parameter = LineParameter(
            dimension,
            {
                "m": memory,
                "max_iterations": max_iterations,
                "max_evaluations": evaluation_limit,
                "past": 0,
                "ftol": 0.0,
                "gtol": 0.0,
            },
        )
        code, final_objective, _ = qn_line(problem, parameter, method, callback=None)
        elapsed = time.perf_counter() - start
        iterations = max_iterations if "MAXIMUMITERATION" in str(code) else None
        exit_status = str(code)
    elif solver == "qnlab qn_ntrqn":
        method = Method(base="NTRQN", store="cautious", secant="damped", update="bfgs")
        parameter = NTRQNParameter(
            dimension,
            {
                "m": memory,
                "max_iterations": max_iterations,
                "max_evaluations": evaluation_limit,
                "past": 0,
                "ftol": 0.0,
                "gtol": 0.0,
            },
        )
        code, final_objective, _ = qn_ntrqn(
            problem, parameter, method, callback=None, verbose=False
        )
        elapsed = time.perf_counter() - start
        iterations = max_iterations if "MAXIMUMITERATION" in str(code) else None
        exit_status = str(code)
    else:
        raise ValueError(f"Unknown solver: {solver}")

    return TimingResult(
        solver=solver,
        dimension=dimension,
        requested_iterations=max_iterations,
        repeat=repeat,
        elapsed_seconds=elapsed,
        iterations=iterations,
        function_calls=problem.call_f,
        gradient_calls=problem.call_g,
        final_objective=float(final_objective),
        exit_status=exit_status,
    )


def benchmark(
    cases: list[tuple[int, int]],
    repeats: int,
    problem_factory=ZeroChainQuadratic,
    memory: int = MEMORY,
) -> pd.DataFrame:
    rows: list[dict] = []
    # Untimed warm-up initializes imports and native code paths.
    for solver in SOLVERS:
        run_once(
            solver,
            dimension=600,
            max_iterations=5,
            repeat=-1,
            memory=memory,
            problem_factory=problem_factory,
        )

    for repeat in range(repeats):
        ordered_solvers = (
            SOLVERS[repeat % len(SOLVERS) :] + SOLVERS[: repeat % len(SOLVERS)]
        )
        for dimension, iterations in cases:
            for solver in ordered_solvers:
                measurement = run_once(
                    solver,
                    dimension,
                    iterations,
                    repeat,
                    memory=memory,
                    problem_factory=problem_factory,
                )
                rows.append(asdict(measurement))
                print(
                    f"repeat={repeat + 1}/{repeats}, n={dimension}, k={iterations}, "
                    f"m={memory}, "
                    f"{solver}: {measurement.elapsed_seconds:.3f} s"
                )
    return pd.DataFrame(rows)

### Fast preflight

This short run catches API or early-termination problems before the expensive measurements. All three rows should report 25 iterations (or the corresponding qnlab iteration-limit status).


In [ ]:
preflight = pd.DataFrame(
    [
        asdict(run_once(solver, dimension=200, max_iterations=25, repeat=0))
        for solver in SOLVERS
    ]
)
display(preflight)
assert preflight["iterations"].eq(25).all(), preflight[
    ["solver", "iterations", "exit_status"]
]

## Experiment 1: elapsed time versus iteration budget

The dimension is fixed at 12,000 while the iteration cap increases to 10,000. Because every point is a fresh run, the plotted time is end-to-end solver time for that cap.


In [ ]:
iteration_cases = [(ITERATION_DIMENSION, int(k)) for k in ITERATION_BUDGETS]
iteration_results = benchmark(iteration_cases, repeats=REPEATS)
iteration_results.to_csv(
    REPO_ROOT / "data" / "SciPy" / "lbfgsb_iteration_scaling.csv", index=False
)
iteration_results

In [ ]:
iteration_summary = iteration_results.groupby(
    ["solver", "requested_iterations"], as_index=False
).agg(
    median_seconds=("elapsed_seconds", "median"),
    min_seconds=("elapsed_seconds", "min"),
    max_seconds=("elapsed_seconds", "max"),
    median_function_calls=("function_calls", "median"),
    median_gradient_calls=("gradient_calls", "median"),
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for solver, group in iteration_summary.groupby("solver"):
    axes[0].plot(
        group["requested_iterations"], group["median_seconds"], "o-", label=solver
    )
    axes[1].plot(
        group["requested_iterations"],
        1e6 * group["median_seconds"] / group["requested_iterations"],
        "o-",
        label=solver,
    )
axes[0].set(
    xlabel="Iteration cap",
    ylabel="Median elapsed time [s]",
    title=f"End-to-end time (n={ITERATION_DIMENSION:,})",
)
axes[1].set(
    xlabel="Iteration cap",
    ylabel="Median time / iteration [µs]",
    title="Normalized iteration cost",
)
for ax in axes:
    ax.grid(True, alpha=0.3)
    ax.legend()
fig.tight_layout()
display(iteration_summary)

## Experiment 2: elapsed time versus dimension

This is the direct test of the suspected more-than-linear growth in $n$. The iteration budget and L-BFGS memory are fixed. A power law $t\approx Cn^p$ is fitted to median wall time only as a descriptive summary. Since fixed Python/setup cost biases small dimensions, inspect the curve and repeat the fit after excluding the smallest points before making a claim. For the implemented $O(n)$ oracle and fixed memory, $p\approx1$ is the natural baseline.


In [ ]:
dimension_cases = [(int(n), DIMENSION_ITERATIONS) for n in DIMENSIONS]
dimension_results = benchmark(dimension_cases, repeats=DIMENSION_REPEATS)
dimension_results.to_csv(
    REPO_ROOT / "data" / "SciPy" / "lbfgsb_dimension_scaling.csv", index=False
)

dimension_summary = dimension_results.groupby(
    ["solver", "dimension"], as_index=False
).agg(
    median_seconds=("elapsed_seconds", "median"),
    min_seconds=("elapsed_seconds", "min"),
    max_seconds=("elapsed_seconds", "max"),
)
slopes = {}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for solver, group in dimension_summary.groupby("solver"):
    slope, intercept = np.polyfit(
        np.log(group["dimension"]), np.log(group["median_seconds"]), 1
    )
    slopes[solver] = slope
    ax.loglog(
        group["dimension"],
        group["median_seconds"],
        "o-",
        label=f"{solver} (p={slope:.2f})",
    )
ax.set(
    xlabel="Dimension n",
    ylabel="Median elapsed time [s]",
    title=f"Dimension scaling ({DIMENSION_ITERATIONS} iterations, m={MEMORY})",
)
ax.grid(True, which="both", alpha=0.3)
ax.legend()
fig.tight_layout()
display(dimension_summary)
pd.Series(slopes, name="log_log_slope_p").to_frame()

## Interpretation checklist

Before opening a SciPy issue, check all of the following:

- SciPy reaches the requested `nit`; each qnlab method exits at its iteration cap.
- Function/gradient counts are comparable enough that line-search work does not explain the timing ratio.
- The effect survives medians of at least three runs and a fresh Python process.
- The dimension-scaling slope is stable when the smallest dimension is omitted.
- The same qualitative result occurs on another machine or SciPy version.
- Profiling identifies time inside SciPy/L-BFGS-B rather than inside this Python objective.

If SciPy is simply slower by a constant factor but has slope near one, report that observation accurately; it is not evidence of superlinear dimension scaling. Objective values are included only as a diagnostic because the algorithms and line searches are not mathematically identical.


## Standalone SciPy-only reproducer

The following cell depends only on the standard library, NumPy, and SciPy and can be pasted into an issue or a separate script. It intentionally omits qnlab. Run it in fresh processes and compare the timings before reporting a regression.


In [ ]:
import time
import numpy as np
from scipy.optimize import minimize


def scipy_only_dimension_scaling(
    dimensions=(1_000, 2_000, 5_000, 10_000, 20_000, 50_000),
    iterations=500,
    repeats=3,
):
    rows = []
    for repeat in range(repeats):
        for n in dimensions:
            x0 = np.zeros(n)

            def fun(x):
                differences = x[:-1] - x[1:]
                return 0.5 * x[0] ** 2 - x[0] + 0.5 * differences @ differences

            def jac(x):
                differences = x[:-1] - x[1:]
                gradient = np.zeros_like(x)
                gradient[0] = x[0] - 1.0
                gradient[:-1] += differences
                gradient[1:] -= differences
                return gradient

            start = time.perf_counter()
            result = minimize(
                fun,
                x0,
                jac=jac,
                method="L-BFGS-B",
                options={
                    "maxcor": 10,
                    "maxiter": iterations,
                    "maxfun": 50 * iterations + 100,
                    "ftol": 0.0,
                    "gtol": 0.0,
                },
            )
            rows.append(
                (
                    repeat,
                    n,
                    time.perf_counter() - start,
                    result.nit,
                    result.nfev,
                    result.message,
                )
            )
    return rows


# Uncomment for the standalone measurement:
# scipy_only_results = scipy_only_dimension_scaling()
# for row in scipy_only_results: print(row)

## Issue-report template

After running the notebook, copy the generated text below and attach this notebook plus the two CSV files. Keep the wording conditional on the actual results. Include the standalone SciPy-only reproducer above and a profiler trace if the anomaly is confirmed.


In [ ]:
scipy_slope = slopes["SciPy L-BFGS-B"]
largest_n = int(DIMENSIONS.max())
largest = dimension_summary[dimension_summary["dimension"] == largest_n].set_index(
    "solver"
)["median_seconds"]
ratio_line = largest["SciPy L-BFGS-B"] / largest["qnlab qn_line"]

issue_text = f"""### Describe the issue
On an analytic-gradient, unconstrained zero-chain quadratic, I measured the wall-clock
scaling of `scipy.optimize.minimize(method='L-BFGS-B')` with `maxcor={MEMORY}`.
At a fixed {DIMENSION_ITERATIONS} iterations, the fitted log-log time/dimension slope was
{scipy_slope:.2f} for SciPy. At n={largest_n:,}, its median wall time was {ratio_line:.2f}x
the median time of the comparison line-search L-BFGS implementation.

This comparison concerns runtime scaling, not solution quality. All methods used an
analytic O(n) objective and gradient, memory {MEMORY}, no callback, and fresh runs.
The attached CSV includes iteration/evaluation counts and exit messages.

### Reproducing code
See the standalone `scipy_only_dimension_scaling` reproducer above and the attached
`SciPy_issue.ipynb` for the controlled comparison.

### Environment
{pd.Series(ENVIRONMENT).to_string()}
"""
print(issue_text)

## Experiment 3: reproduce `script/computation_time.py` at large dimension

This additional experiment stays close to `script/computation_time.py`: it uses `IllQuadraticProblem`, L-BFGS memory $m=10$, and 100 iterations. The original script fixes $n=10{,}000$, runs 100 repeats, and includes all methods returned by `get_methods`; here we retain only the three methods relevant to this investigation, sweep through $n=100{,}000$, and use 10 repeats by default.

The diagonal quadratic is

$$f(x)=\frac12 x^T\operatorname{diag}(1,\ldots,n)x,\qquad x_0=(1,\ldots,1).$$

Unlike the zero-chain experiment, this section is primarily a fixed-work, large-array runtime test. Function/gradient counts and exit statuses remain part of the output so that a timing difference is not mistaken for different work.


In [ ]:
from qnlab.problem.ill_quadratic import IllQuadraticProblem

ILL_QUADRATIC_ITERATIONS = 1000
ILL_QUADRATIC_DIMENSIONS = np.array([1_000, 3_000, 10_000, 30_000, 100_000, 300_000])
ILL_QUADRATIC_REPEATS = 10  # Use 3 for a quick run; the original script uses 100.

ill_quadratic_cases = [
    (int(n), ILL_QUADRATIC_ITERATIONS) for n in ILL_QUADRATIC_DIMENSIONS
]
ill_quadratic_results = benchmark(
    ill_quadratic_cases,
    repeats=ILL_QUADRATIC_REPEATS,
    problem_factory=IllQuadraticProblem,
)
ill_quadratic_results.to_csv(
    REPO_ROOT / "data" / "SciPy" / "lbfgsb_ill_quadratic_scaling.csv", index=False
)
ill_quadratic_results

In [ ]:
ill_quadratic_summary = ill_quadratic_results.groupby(
    ["solver", "dimension"], as_index=False
).agg(
    median_seconds=("elapsed_seconds", "median"),
    min_seconds=("elapsed_seconds", "min"),
    max_seconds=("elapsed_seconds", "max"),
    median_function_calls=("function_calls", "median"),
    median_gradient_calls=("gradient_calls", "median"),
    median_objective=("final_objective", "median"),
)
reference_times = ill_quadratic_summary[
    ill_quadratic_summary["solver"] == "qnlab qn_line"
].set_index("dimension")["median_seconds"]
ill_quadratic_summary["ratio_to_qn_line"] = ill_quadratic_summary.apply(
    lambda row: row["median_seconds"] / reference_times.loc[row["dimension"]], axis=1
)

ill_quadratic_slopes = {}
ill_quadratic_large_n_slopes = {}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for solver, group in ill_quadratic_summary.groupby("solver"):
    slope = np.polyfit(np.log(group["dimension"]), np.log(group["median_seconds"]), 1)[
        0
    ]
    large_group = group[group["dimension"] >= 30_000]
    large_slope = np.polyfit(
        np.log(large_group["dimension"]), np.log(large_group["median_seconds"]), 1
    )[0]
    ill_quadratic_slopes[solver] = slope
    ill_quadratic_large_n_slopes[solver] = large_slope
    axes[0].loglog(
        group["dimension"],
        group["median_seconds"],
        "o-",
        label=f"{solver} (p={slope:.2f})",
    )
    axes[1].semilogx(group["dimension"], group["ratio_to_qn_line"], "o-", label=solver)
axes[0].set(
    xlabel="Dimension n",
    ylabel="Median elapsed time [s]",
    title=f"Ill-conditioned quadratic ({ILL_QUADRATIC_ITERATIONS} iterations)",
)
axes[1].axhline(1.0, color="black", linewidth=1, alpha=0.5)
axes[1].set(
    xlabel="Dimension n",
    ylabel="Median time / qn_line median time",
    title="Relative runtime",
)
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
fig.tight_layout()
display(ill_quadratic_summary)
display(
    pd.DataFrame(
        {
            "all_dimensions_slope": ill_quadratic_slopes,
            "n_at_least_30000_slope": ill_quadratic_large_n_slopes,
        }
    )
)

### How to interpret the large-quadratic result

Pay particular attention to the relative-runtime panel around $n=100{,}000$. A localized peak means SciPy is slower at that size, but does **not** by itself establish asymptotically superlinear scaling. For that claim, the SciPy curve and its large-$n$ slope should continue to separate from both qnlab curves as $n$ grows. Abrupt changes shared by all methods are more plausibly caused by cache, memory bandwidth, BLAS threading, CPU frequency, or allocation effects.

For an issue report, repeat the experiment in fresh processes and record thread-related environment variables such as `OMP_NUM_THREADS`, `OPENBLAS_NUM_THREADS`, and `MKL_NUM_THREADS`. If the slowdown is localized, profile the $n=100{,}000$ case and compare it with neighboring dimensions rather than describing it as a general complexity result.


In [ ]:
n_of_interest = 100_000
at_100k = ill_quadratic_summary[
    ill_quadratic_summary["dimension"] == n_of_interest
].set_index("solver")
scipy_vs_line_100k = (
    at_100k.loc["SciPy L-BFGS-B", "median_seconds"]
    / at_100k.loc["qnlab qn_line", "median_seconds"]
)
scipy_vs_ntrqn_100k = (
    at_100k.loc["SciPy L-BFGS-B", "median_seconds"]
    / at_100k.loc["qnlab qn_ntrqn", "median_seconds"]
)
print(
    f"At n={n_of_interest:,}, SciPy/qn_line={scipy_vs_line_100k:.2f}x and "
    f"SciPy/qn_ntrqn={scipy_vs_ntrqn_100k:.2f}x. "
    f"SciPy's fitted slope is {ill_quadratic_slopes['SciPy L-BFGS-B']:.2f} over all "
    f"dimensions and {ill_quadratic_large_n_slopes['SciPy L-BFGS-B']:.2f} for n>=30,000."
)

## Experiment 4: L-BFGS memory $m=3$ versus $m=10$

This experiment repeats the large diagonal-quadratic comparison with two memory sizes. For SciPy, `memory` is passed as `maxcor`; for `qn_line` and `qn_ntrqn`, the same value is passed as `m`. Thus each curve compares like-for-like limited-memory storage, although the three solvers still use different globalization strategies.

All `(memory, dimension, solver)` cases are shuffled within each repeat to reduce ordering and CPU-frequency bias. The primary diagnostic is the ratio `median time at m=3 / median time at m=10`: values below one mean that the smaller memory is faster. Function/gradient counts and final objectives remain visible because changing memory may also change line-search work and optimization progress.

In [ ]:
from qnlab.problem.ill_quadratic import IllQuadraticProblem

MEMORY_VALUES = (3, 10)
MEMORY_COMPARISON_REPEATS = 5  # Use 3 for a quick run.
MEMORY_COMPARISON_DIMENSIONS = np.array(
    [1_000, 3_000, 10_000, 30_000, 100_000, 300_000]
)
MEMORY_COMPARISON_ITERATIONS = 1_000

# Untimed warm-up for every solver-memory combination.
for memory in MEMORY_VALUES:
    for solver in SOLVERS:
        run_once(
            solver,
            dimension=600,
            max_iterations=5,
            repeat=-1,
            memory=memory,
            problem_factory=IllQuadraticProblem,
        )

memory_cases = [
    (memory, int(dimension), solver)
    for memory in MEMORY_VALUES
    for dimension in MEMORY_COMPARISON_DIMENSIONS
    for solver in SOLVERS
]
memory_rows = []
for repeat in range(MEMORY_COMPARISON_REPEATS):
    order = np.random.default_rng(20260726 + repeat).permutation(len(memory_cases))
    for case_index in order:
        memory, dimension, solver = memory_cases[case_index]
        measurement = run_once(
            solver,
            dimension,
            MEMORY_COMPARISON_ITERATIONS,
            repeat,
            memory=memory,
            problem_factory=IllQuadraticProblem,
        )
        row = asdict(measurement)
        row["memory"] = memory
        memory_rows.append(row)
        print(
            f"repeat={repeat + 1}/{MEMORY_COMPARISON_REPEATS}, "
            f"n={dimension}, m={memory}, {solver}: "
            f"{measurement.elapsed_seconds:.3f} s"
        )

memory_results = pd.DataFrame(memory_rows)
memory_results.to_csv(
    REPO_ROOT / "data" / "SciPy" / "lbfgsb_memory_3_vs_10.csv", index=False
)
memory_results

In [ ]:
memory_summary = memory_results.groupby(
    ["solver", "memory", "dimension"], as_index=False
).agg(
    median_seconds=("elapsed_seconds", "median"),
    min_seconds=("elapsed_seconds", "min"),
    max_seconds=("elapsed_seconds", "max"),
    median_function_calls=("function_calls", "median"),
    median_gradient_calls=("gradient_calls", "median"),
    median_objective=("final_objective", "median"),
)

memory_ratio = memory_summary.pivot(
    index=["solver", "dimension"], columns="memory", values="median_seconds"
).reset_index()
memory_ratio["m3_over_m10"] = memory_ratio[3] / memory_ratio[10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.3))
for (solver, memory), group in memory_summary.groupby(["solver", "memory"]):
    axes[0].loglog(
        group["dimension"],
        group["median_seconds"],
        "o-",
        label=f"{solver}, m={memory}",
    )
for solver, group in memory_ratio.groupby("solver"):
    axes[1].semilogx(group["dimension"], group["m3_over_m10"], "o-", label=solver)
axes[0].set(
    xlabel="Dimension n",
    ylabel="Median elapsed time [s]",
    title=f"Memory comparison ({MEMORY_COMPARISON_ITERATIONS} iterations)",
)
axes[1].axhline(1.0, color="black", linewidth=1, alpha=0.5)
axes[1].set(
    xlabel="Dimension n",
    ylabel="Median time at m=3 / median time at m=10",
    title="Effect of reducing L-BFGS memory",
)
for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
    ax.legend(fontsize="small")
fig.tight_layout()
display(memory_summary)
display(memory_ratio)

### Interpreting the memory comparison

A smaller `maxcor` reduces the amount of vector work and storage inside L-BFGS-B, so an `m=3` speedup that grows with dimension is plausible. It is not automatically evidence that `m=10` has a complexity bug: compare evaluation counts and final objectives first. If those are similar and the ratio still changes sharply around a particular dimension, profiling both memory settings at that dimension can help separate the expected $O(nm)$ work from allocation, cache, or wrapper overhead.